# 08. Predicting posts piece by piece

**Hypothesis:** the models were trained on single sentences, but competition posts are much longer and contain line breaks and emoji bullets. If each post is cut into lines (or sentences), every piece is predicted separately, and the location strings are joined back, the model sees input that looks like its training data. This changes only inference: the three fold models from notebook 06 are used as they are, and nothing is retrained.

**Three ways to feed a post** (`nerkit/chunked.py`):
- `none`: the whole post at once (what notebooks 06 and 07 did).
- `lines`: every non-empty line separately.
- `sentences`: every line, cut further after `.` `!` `?` when a capital letter follows; abbreviations such as `вул.`, `м.` and initials are not cut.

**What can and cannot be checked here**
- *Synthetic posts* (held-out sentences joined into multi-line posts, as in notebook 04) are scored per fold with each fold's own held-out text, so there is no leak. They show whether long inputs cost accuracy and whether splitting wins it back. The posts are built from sentences, so splitting by lines is an easy case here; the splitter is still a plain rule and not told where the joins are.
- The 26-post *labeling sample* is real competition text but tiny and noisy.
- The real test is the leaderboard, so the notebook also writes submission files.

In [1]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive/ai301")
    ROOT = PROJECT / "deep-learning-assignment3"
    BRANCH = "model"  # the branch that holds model/; main does not have it
    if not ROOT.exists():
        PROJECT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "-b", BRANCH, "https://github.com/koalla05/deep-learning-assignment3.git", str(ROOT)], check=True)
    else:
        subprocess.run(["git", "fetch", "origin", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "checkout", BRANCH], cwd=ROOT, check=True)
        subprocess.run(["git", "pull", "origin", BRANCH], cwd=ROOT, check=True)
    # The frozen split manifests are stored with Git LFS; a plain clone only has small pointer files.
    subprocess.run("apt-get install -y -q git-lfs", shell=True)
    subprocess.run(["git", "lfs", "install"], cwd=ROOT)
    subprocess.run(["git", "lfs", "pull"], cwd=ROOT)
    import urllib.request

    for name in ("role_manifest_v2.parquet", "cv_manifest_v1.parquet"):
        manifest = ROOT / "eda" / "output" / name
        if manifest.read_bytes()[:4] != b"PAR1":  # still a pointer file: download the real one
            url = f"https://media.githubusercontent.com/media/koalla05/deep-learning-assignment3/{BRANCH}/eda/output/{name}"
            urllib.request.urlretrieve(url, manifest)
        assert manifest.read_bytes()[:4] == b"PAR1", f"{name} is not a parquet file; download it from GitHub and upload it to {manifest.parent}"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "lightning", "pytorch-crf", "tokenizers"], check=True)
    os.environ["NER_DATA_DIR"] = str(PROJECT / "data" / "kse-ua-location-extraction-2026")
    os.environ["NER_WORK_DIR"] = str(PROJECT / "ner_work")
else:
    ROOT = Path.cwd()
    while not (ROOT / "eda").exists():
        ROOT = ROOT.parent
    # Results downloaded from Colab (folder ner_work) are continued here: finished folds are skipped.
    if "NER_WORK_DIR" not in os.environ and (ROOT / "data" / "ner_work").exists():
        os.environ["NER_WORK_DIR"] = str(ROOT / "data" / "ner_work")

import torch

os.environ.setdefault("NER_WANDB", "off")
os.environ["NER_DEVICE"] = "cuda" if torch.cuda.is_available() else "cpu"
sys.path[:0] = [str(ROOT), str(ROOT / "model")]
print("Device:", os.environ["NER_DEVICE"], "| project:", ROOT, "| work folder:", os.environ.get("NER_WORK_DIR", ROOT / "model"))

Device: cpu | project: /Users/allakovalcuk/AI301/deep-learning-assignment3 | work folder: /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work


In [2]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from nerkit.artifacts import combine_models, load_logits, majority_vote, row_key, write_submission
from nerkit.chunked import MODES, decode_chunks, infer, load_trained, make_chunks, split_text
from nerkit.data import load_labeling_sample, load_rows, load_test
from nerkit.decisions import DECODING
from nerkit.experiment import evaluation_sets, read_results, select_training
from nerkit.config import WORK_DIR
from nerkit.metrics import evaluate

SOURCE = "06_final_cv_training"
DECODE = {name: DECODING[name] for name in ("merge", "drop_empty", "trim_number", "bias")}
records = sorted(read_results(SOURCE, raw=True), key=lambda record: record["config"]["fold"])
assert len(records) == 3, "run notebook 06 first"
trained = [load_trained(record) for record in records]
print("Loaded fold models:", [record["key"] for record in records])

Loaded fold models: ['367d29e13b0b', '60d6b400897e', 'c048151b9c64']


## How different is the input from the training data?

Length in words of single training sentences, whole test posts, and the pieces after splitting.

In [3]:
test_rows = load_test()
labeling_rows = load_labeling_sample()
train_texts = [row["text"] for row in load_rows(("cv",))][:20_000]

def length_row(texts):
    words = np.array([len(text.split()) for text in texts])
    return dict(texts=len(words), median_words=np.median(words), mean_words=words.mean(), p90_words=np.percentile(words, 90))

groups = {"training sentences": train_texts, "test posts (whole)": [row["text"] for row in test_rows]}
for mode in ("lines", "sentences"):
    groups[f"test pieces ({mode})"] = [row["text"][a:b] for row in test_rows for a, b in split_text(row["text"], mode)]
display(pd.DataFrame({name: length_row(texts) for name, texts in groups.items()}).T.round(1))
share = np.mean(["\n" in row["text"] for row in test_rows])
print(f"{share:.0%} of the test posts contain line breaks; the training sentences contain none.")

,texts,median_words,mean_words,p90_words
training sentences,20000.0,15.0,16.0,26.0
test posts (whole),477.0,36.0,54.9,118.0
test pieces (lines),1859.0,10.0,14.1,30.0
test pieces (sentences),2502.0,9.0,10.5,20.0


85% of the test posts contain line breaks; the training sentences contain none.


## Check that the rebuilt models reproduce the saved logits

The models are rebuilt from `tokenizer.json` and `model.pt`. Predicting 100 whole test posts must give the same logits as the saved `test.npz` (up to float16 rounding).

In [4]:
sample = test_rows[:100]
saved, _ = load_logits(records[0]["key"], "test")
chunks, owners = make_chunks(sample, "none")
fresh = infer(trained[0], chunks)
difference = max(np.abs(fresh[i][1] - saved[row_key(row)][1]).max() for i, row in enumerate(sample))
print(f"largest difference to the saved logits: {difference:.4f}")
assert difference < 0.1, "the rebuilt model does not match the saved one"

largest difference to the saved logits: 0.0039


## Synthetic posts, per fold

Each fold model is scored on posts built from its own held-out fold. `heldout sentences` is the same model on the single sentences, i.e. what it can reach when the input looks like its training data.

In [5]:
rows = []
for record, model in zip(records, trained):
    config = model["config"]
    _, _, _, heldout = select_training(config)
    posts = evaluation_sets(config, heldout)["posts"]
    entry = {"fold": config.fold, "heldout sentences": record["scores"]["heldout"]["mix"]["f1"]}
    for mode in MODES:
        chunks, owners = make_chunks(posts, mode)
        predictions = decode_chunks(infer(model, chunks), chunks, owners, len(posts), model["crf"], **DECODE)
        entry[mode] = evaluate(posts, predictions)["mix"]["f1"]
    rows.append(entry)
synthetic = pd.DataFrame(rows).set_index("fold")
display(synthetic.round(3))
display(synthetic.mean().round(3).to_frame("mean over folds").T)

,heldout sentences,none,lines,sentences
fold,,,,
0,0.885,0.863,0.882,0.882
1,0.896,0.873,0.888,0.888
2,0.894,0.882,0.892,0.892


,heldout sentences,none,lines,sentences
mean over folds,0.891,0.873,0.887,0.887


## The 26-post labeling sample

Real competition posts that no model saw. Too small to choose between methods, but a first look at real text.

In [6]:
def predict_all(rows, mode):
    """Strings per fold model, for the average of the three models, and for a 2-of-3 vote."""
    chunks, owners = make_chunks(rows, mode)
    dicts = [infer(model, chunks) for model in trained]
    texts = {position: chunk["text"] for position, chunk in enumerate(chunks)}
    folds = [decode_chunks(loaded, chunks, owners, len(rows), model["crf"], **DECODE) for loaded, model in zip(dicts, trained)]
    combined, crf = combine_models(dicts, [{"crf": model["crf"]} for model in trained], texts)
    average = decode_chunks(combined, chunks, owners, len(rows), crf, **DECODE)
    return {**{f"fold {i}": values for i, values in enumerate(folds)}, "average of 3": average, "vote, 2 of 3": majority_vote(folds, 2)}

labeling_scores, precision_recall = {}, {}
for mode in MODES:
    predictions = predict_all(labeling_rows, mode)
    scored = {name: evaluate(labeling_rows, values)["pooled"] for name, values in predictions.items()}
    labeling_scores[mode] = {name: score["f1"] for name, score in scored.items()}
    precision_recall[mode] = {"precision": scored["average of 3"]["precision"], "recall": scored["average of 3"]["recall"],
                              "locations predicted": sum(len(values) for values in predictions["average of 3"])}
labeling_table = pd.DataFrame(labeling_scores)
display(labeling_table.round(3))
print("Average of the three models, by mode:")
display(pd.DataFrame(precision_recall).T.round(3))

,none,lines,sentences
fold 0,0.606,0.537,0.567
fold 1,0.722,0.563,0.592
fold 2,0.676,0.556,0.583
average of 3,0.694,0.551,0.580
"vote, 2 of 3",0.704,0.571,0.600


Average of the three models, by mode:


,precision,recall,locations predicted
none,0.625,0.781,40.0
lines,0.514,0.594,37.0
sentences,0.541,0.625,37.0


## Test predictions

Every fold model predicts the 477 posts in each mode. The `none` row must agree with the CSVs submitted earlier, which confirms the pipeline. Files for `lines` and `sentences` are written for the average of the three models and for the 2-of-3 vote.

In [7]:
import csv

test_predictions = {mode: predict_all(test_rows, mode) for mode in MODES}

def earlier(fold):
    path = WORK_DIR / "submissions" / f"final_fold{fold}.csv"
    with path.open(newline="", encoding="utf-8") as handle:
        return [json.loads(row["locations"]) for row in csv.DictReader(handle)]

for fold in range(3):
    same = np.mean([set(a) == set(b) for a, b in zip(test_predictions["none"][f"fold {fold}"], earlier(fold))])
    print(f"mode none, fold {fold}: {same:.1%} of the posts equal the file submitted earlier")

summary = {}
for mode in MODES:
    for name in ("fold 0", "fold 1", "fold 2", "average of 3", "vote, 2 of 3"):
        values = test_predictions[mode][name]
        summary[(mode, name)] = dict(posts_with_locations=sum(bool(v) for v in values), locations=sum(len(v) for v in values))
display(pd.DataFrame(summary).T)

for mode in ("lines", "sentences"):
    for name, label in (("average of 3", "average"), ("vote, 2 of 3", "vote2")):
        path = write_submission(WORK_DIR / "submissions" / f"split_{mode}_{label}.csv", test_rows, test_predictions[mode][name])
        print("wrote", path.name)

mode none, fold 0: 100.0% of the posts equal the file submitted earlier
mode none, fold 1: 100.0% of the posts equal the file submitted earlier
mode none, fold 2: 100.0% of the posts equal the file submitted earlier


posts_with_locations  locations
none      fold 0                         317        701
          fold 1                         333        749
          fold 2                         326        758
          average of 3                   322        734
          vote, 2 of 3                   323        712
lines     fold 0                         327        739
          fold 1                         339        773
          fold 2                         336        791
          average of 3                   333        766
          vote, 2 of 3                   331        746
sentences fold 0                         329        744
          fold 1                         342        778
          fold 2                         339        796
          average of 3                   334        769
          vote, 2 of 3                   332        750

wrote split_lines_average.csv
wrote split_lines_vote2.csv
wrote split_sentences_average.csv
wrote split_sentences_vote2.csv


## Saved for reuse

In [8]:
package = WORK_DIR / "final_cv"
package.mkdir(parents=True, exist_ok=True)
(package / "split_inference_summary.json").write_text(json.dumps(dict(
    synthetic_posts_f1=synthetic.round(4).to_dict(), labeling_f1=labeling_table.round(4).to_dict(),
    test_counts={f"{mode}|{name}": counts for (mode, name), counts in summary.items()}), indent=1), encoding="utf-8")
print("Saved", package / "split_inference_summary.json")

Saved /Users/allakovalcuk/AI301/deep-learning-assignment3/data/ner_work/final_cv/split_inference_summary.json


## Leaderboard results and conclusion

Public scores of the two split files that were uploaded, next to the same 2-of-3 vote on whole posts (notebook 07).

**Conclusion:** splitting posts into lines or sentences lowers the leaderboard score by about 0.015, which matches the labeling sample (also lower) and not the synthetic posts (higher). Cutting a post into pieces helps when the "post" is glued-together training sentences, but it hurts on real posts, where the surrounding lines help the model decide what is a location. Whole-post prediction stays the final choice, and the length mismatch is not what limits the score.

In [9]:
import pandas as pd
from IPython.display import display

LEADERBOARD_SPLIT = {"whole posts, vote 2 of 3 (notebook 07)": 0.58914,
                     "split_lines_vote2.csv": 0.57228,
                     "split_sentences_vote2.csv": 0.57488}  # public scores from Kaggle
board = pd.Series(LEADERBOARD_SPLIT, name="public_f1")
display(board.to_frame().round(5))
reference = board.iloc[0]
for name, score in board.iloc[1:].items():
    print(f"{name}: {score - reference:+.4f} vs whole posts")

,public_f1
"whole posts, vote 2 of 3 (notebook 07)",0.58914
split_lines_vote2.csv,0.57228
split_sentences_vote2.csv,0.57488


split_lines_vote2.csv: -0.0169 vs whole posts
split_sentences_vote2.csv: -0.0143 vs whole posts
